In [1]:
import os
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [2]:
import pandas as pd

In [3]:
UCR_128 = ['ACSF1', 'Adiac', 'AllGestureWiimoteX', 'AllGestureWiimoteY', 'AllGestureWiimoteZ', 'ArrowHead', 'BME', 'Beef', 'BeetleFly', 'BirdChicken', 'CBF', 'Car', 'Chinatown', 'ChlorineConcentration', 'CinCECGTorso', 'Coffee', 'Computers', 'CricketX', 'CricketY', 'CricketZ', 'Crop', 'DiatomSizeReduction', 'DistalPhalanxOutlineAgeGroup', 'DistalPhalanxOutlineCorrect', 'DistalPhalanxTW', 'DodgerLoopDay', 'DodgerLoopGame', 'DodgerLoopWeekend', 'ECG200', 'ECG5000', 'ECGFiveDays', 'EOGHorizontalSignal', 'EOGVerticalSignal', 'Earthquakes', 'ElectricDevices', 'EthanolLevel', 'FaceAll', 'FaceFour', 'FacesUCR', 'FiftyWords', 'Fish', 'FordA', 'FordB', 'FreezerRegularTrain', 'FreezerSmallTrain', 'Fungi', 'GestureMidAirD1', 'GestureMidAirD2', 'GestureMidAirD3', 'GesturePebbleZ1', 'GesturePebbleZ2', 'GunPoint', 'GunPointAgeSpan', 'GunPointMaleVersusFemale', 'GunPointOldVersusYoung', 'Ham', 'HandOutlines', 'Haptics', 'Herring', 'HouseTwenty', 'InlineSkate', 'InsectEPGRegularTrain', 'InsectEPGSmallTrain', 'InsectWingbeatSound', 'ItalyPowerDemand', 'LargeKitchenAppliances', 'Lightning2', 'Lightning7', 'Mallat', 'Meat', 'MedicalImages', 'MelbournePedestrian', 'MiddlePhalanxOutlineAgeGroup', 'MiddlePhalanxOutlineCorrect', 'MiddlePhalanxTW', 'MixedShapesRegularTrain', 'MixedShapesSmallTrain', 'MoteStrain', 'NonInvasiveFetalECGThorax1', 'NonInvasiveFetalECGThorax2', 'OSULeaf', 'OliveOil', 'PLAID', 'PhalangesOutlinesCorrect', 'Phoneme', 'PickupGestureWiimoteZ', 'PigAirwayPressure', 'PigArtPressure', 'PigCVP', 'Plane', 'PowerCons', 'ProximalPhalanxOutlineAgeGroup', 'ProximalPhalanxOutlineCorrect', 'ProximalPhalanxTW', 'RefrigerationDevices', 'Rock', 'ScreenType', 'SemgHandGenderCh2', 'SemgHandMovementCh2', 'SemgHandSubjectCh2', 'ShakeGestureWiimoteZ', 'ShapeletSim', 'ShapesAll', 'SmallKitchenAppliances', 'SmoothSubspace', 'SonyAIBORobotSurface1', 'SonyAIBORobotSurface2', 'StarLightCurves', 'Strawberry', 'SwedishLeaf', 'Symbols', 'SyntheticControl', 'ToeSegmentation1', 'ToeSegmentation2', 'Trace', 'TwoLeadECG', 'TwoPatterns', 'UMD', 'UWaveGestureLibraryAll', 'UWaveGestureLibraryX', 'UWaveGestureLibraryY', 'UWaveGestureLibraryZ', 'Wafer', 'Wine', 'WordSynonyms', 'Worms', 'WormsTwoClass', 'Yoga']

In [ ]:
ckpt_dir = '/data/yoom618/TSLib/checkpoints/'
best_ckpt_dir = '/data/yoom618/TSLib/checkpoints_final/'
best_script_dir = '/data/yoom618/TSLib/scripts_final/'

model = 'MambaSL'
for subdir, exp in [
    ('additional (ucr, inceptiontime-setting)', 'trainlossonly'),
    ]:
    print(f'==================={model}===================')
    os.makedirs(os.path.join(best_ckpt_dir, model, subdir), exist_ok=True)

    for data_name in UCR_128:

        if os.path.exists(f'../03-full_results/{model}/{subdir}/{model}_UCR_{data_name}_{exp}.out'):
            with open(f'../03-full_results/{model}/{subdir}/{model}_UCR_{data_name}_{exp}.out', 'r') as file:
                data = file.read().splitlines()
            with open(f'../03-full_results/{model}/{subdir}/{model}_UCR_{data_name}_{exp}.sh', 'r') as file:
                scripts = file.read().split('\n\n')[:-1]
                scripts = list(filter(lambda x: x[0] != '#', scripts))
        else:
            # print('no file')
            continue


        print(data_name)
        result_lst = list()
        for i in range(len(data)):
            if data[i].startswith('>>>>>>>testing : '):
                ckpt = data[i].replace('>>>>>>>testing : ', '').replace('<', '').strip()
                data_meta = ckpt.split('_')

                if data[i+2].startswith('gating_value'):
                    acc = float(data[i+3].replace('accuracy:', ''))
                    model_params = data[i+4].replace('model parameter : ', '')
                    model_size = data[i+5].replace('model size : ', '').replace('MB', '')
                else:
                    acc = float(data[i+2].replace('accuracy:', ''))
                    model_params = data[i+3].replace('model parameter : ', '')
                    model_size = data[i+4].replace('model size : ', '').replace('MB', '')

                result_data = {
                    i: data_meta[i] for i in range(9, len(data_meta))
                }
                result_data.update({
                    'acc': float(acc),
                    'model_params': int(model_params),
                    'model_size (MB)': float(model_size),
                    'ckpt': ckpt
                })

                result_lst.append(result_data)

        result_df = pd.DataFrame(reversed(result_lst))
        scripts = list(reversed(scripts))

        if len(result_lst) < 200:
            continue
        
        print(result_df['acc'].max() * 100, len(result_df))
        result_acc_max = result_df[result_df['acc'] == result_df['acc'].max()].sort_values(['model_size (MB)', 'model_params'])
        # display(result_acc_max)
        result_acc_max5 = result_df[result_df['acc'] >= result_df['acc'].nlargest(5).min()].sort_values(['acc', 'model_size (MB)', 'model_params'], ascending=[False, True, True])



        # select best accuracy model (up to 20)
        result_acc_max = result_acc_max.iloc[:20]

        # move 20 lowest sized ckpts
        for ckpt in result_acc_max['ckpt']:
            if ckpt in os.listdir(ckpt_dir):
                os.makedirs(os.path.join(best_ckpt_dir, model, f'{subdir}-max'), exist_ok=True)
                os.rename(os.path.join(ckpt_dir, ckpt), os.path.join(best_ckpt_dir, model, f'{subdir}-max', ckpt))
                print(f'> moved {ckpt}')
        
        for ckpt in result_acc_max5['ckpt']:
            if ckpt in os.listdir(ckpt_dir):
                os.makedirs(os.path.join(best_ckpt_dir, model, f'{subdir}-top5'), exist_ok=True)
                os.rename(os.path.join(ckpt_dir, ckpt), os.path.join(best_ckpt_dir, model, f'{subdir}-top5', ckpt))
                print(f'> moved {ckpt} (top5 acc)')
                

        # assert len(result_df) == len(scripts), f'len(result_df) == len(scripts ({len(result_df)} != {len(scripts)})'
        if len(result_df) != len(scripts):
            print(f'{data_name} : len(result_df) != len(scripts) ({len(result_df)} != {len(scripts)})')

        # make the scripts
        scripts_acc_max = f'''model_name="MambaSingleLayer"
dataset_name="{data_name}"
tslib_dir="/data/yoom618/TSLib"
gpu_id=0

data_dir="${{tslib_dir}}/dataset_UCR"
checkpoint_dir="${{tslib_dir}}/checkpoints_best/${{model_name}} (UCR, inceptiontime-setting)"'''
        if len(result_acc_max) > 1:
            scripts_acc_max += '\n\n# below all have the same performance'

        for idx in result_acc_max.index:
            scripts_acc_max += '''\n\npython run.py \\
  --use_gpu True \\
  --gpu_type cuda \\
  --gpu ${gpu_id} \\
  --task_name classification_trainlossonly \\
  --data UEA \\
  --root_path "${data_dir}/${dataset_name}" \\
  --checkpoints "${checkpoint_dir}" \\
  --model ${model_name} \\
  --model_id "${dataset_name}" \\
'''
            scripts_acc_max += '\n'.join(scripts[idx].replace("--is_training 1", "--is_training 0").split('\n')[15:])
        scripts_acc_max = scripts_acc_max.strip()
        out_dir = os.path.join(best_script_dir, model, subdir)
        script_fname = f'{out_dir}/{data_name}.sh'
        if not os.path.exists(script_fname):
            os.makedirs(out_dir, exist_ok=True)
            with open(script_fname, 'w') as file:
                file.write(scripts_acc_max)
            print(f'> saved {data_name}.sh')
            print()
    
        
        